# 5.3 Two-Dimensional Gratings and Diffraction-Order Lattices

A grating that repeats in one direction gives a row of candidate orders. Repeating the pattern in a second independent direction gives a two-dimensional array. Each order now needs two integers. We will construct this array of wavevectors, called an order lattice, using the coordinates we already know.

[5.1](5_1_Three_Dimensional_Orders_and_Conical_Diffraction.ipynb) gave one order integer even for conical incidence. [5.2](5_2_Direction_Cosines_and_Spatial_Frequency_Coordinates.ipynb) gave equivalent wavevector, frequency and direction-cosine representations. We now change the structure's symmetry, keeping those coordinates.

:::{note} Model and wavelength conventions

Run independently in order with NumPy and Matplotlib. The grating lies at $z=0$; use $e^{-i\omega t}$ and transmission toward $-z$. Lengths are in micrometres. Here $\lambda_0$ is vacuum wavelength, $k_0=2\pi/\lambda_0$, and positive real $n_{\rm in},n_{\rm out}$ specify lossless exterior media. The earlier explorers retain their in-medium wavelength controls. No objective is introduced in this lesson. Complex excitation is developed later in [7.4](7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb); it is not a prerequisite.

:::

In [ ]:
from pathlib import Path
import sys, importlib
_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p/'wave_optics'/'__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
import wave_optics
wave_optics.__path__[:] = [str(_support_dir/'wave_optics')]
from wave_optics import lattice_grating as lg
importlib.reload(lg)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from wave_optics.lattice_plotting import coordinate_plot


## 1. Two translation symmetries, two integers

Move by one period in either direction. In each case, the outgoing field must reproduce the phase change imposed by the incident wave, allowing for complete extra cycles. There are now two phase-matching conditions. A periodic transmission function lets us write them explicitly. For $t(x+p_x,y)=t(x,y)$ and $t(x,y+p_y)=t(x,y)$, expand
$$t(x,y)=\sum_{m,p\in\mathbb Z}t_{mp}\exp\!\left[i2\pi\left(\frac{mx}{p_x}+\frac{py}{p_y}\right)\right].$$
The incident surface field is $U_{\rm inc}(x,y,0)=A_{\rm inc}e^{i\mathbf k_{\parallel,\rm inc}\cdot(x,y)}$, with
$$k_0=2\pi/\lambda_0,\qquad \mathbf k_{\parallel,\rm inc}=n_{\rm in}k_0(\alpha_{\rm inc},\beta_{\rm inc}).$$
Here $\alpha_{\rm inc}^2+\beta_{\rm inc}^2<1$. Multiplying by $t$ adds the exponents, giving
$$\boxed{\mathbf k_{\parallel,mp}=\mathbf k_{\parallel,\rm inc}+2\pi(m/p_x,p/p_y)}.$$
Each integer pair identifies a possible harmonic. Its coefficient may still be zero, and it may be evanescent. Changing the incident direction shifts the whole array of candidate wavevectors. Changing either pitch changes the corresponding spacing. Those reciprocal spacings are set by the periods, rather than the refractive indices.

A conically illuminated **one-dimensional** grating still has uniform grooves: $k_y=k_{y,\rm inc}$ and only one order integer. Two-dimensional periodicity permits discrete changes in both transverse components. There is generally no common groove-axis cone.

For direct vectors $\mathbf a_1,\mathbf a_2$ and reciprocal vectors $\mathbf b_1,\mathbf b_2$,
$$\mathbf a_i\cdot\mathbf b_j=2\pi\delta_{ij},\qquad
\mathbf k_{\parallel,mp}=\mathbf k_{\parallel,\rm inc}+m\mathbf b_1+p\mathbf b_2.$$
With direct vectors as columns of $A$, $B=2\pi A^{-T}$. A rigid in-plane rotation $R$ sends $A\to RA$, $B\to RB$. We keep rectangular examples below.



Equivalently, translating the incident and outgoing surface fields by $p_x$ and $p_y$ requires
$$e^{i(k_x-k_{x,\rm inc})p_x}=1,\qquad e^{i(k_y-k_{y,\rm inc})p_y}=1.$$
The two phase differences are independently $2\pi m$ and $2\pi p$. This symmetry rule also constrains a physical periodic scattering problem; the simple multiplication by $t$ is only a thin-mask prescription for its coefficients.

### A generic rectangular grating

Use $p_x=4$, $p_y=5$, and $\lambda_0=2$ in exterior index 1. Oblique illumination has $(\alpha_{\rm inc},\beta_{\rm inc})=(0.3,0.4)$, as in the conical example. The drawing shows centered rectangular clear openings with fractions $(1/2,3/8)$ solely to illustrate the periodic structure. No amplitudes are inferred from this drawing.

Here $m,p$ are order integers, while $p_x,p_y$ are physical periods. The code retains `n`/`nn` for the second order; these identifiers correspond to $p$, not refractive index.


In [ ]:
wavelength0 = 2.
px, py = 4., 5.
Dx, Dy = .5, .375
A = np.diag([px, py])
v = np.arange(-6, 7)
mm, nn = np.meshgrid(v, v)
labels = np.column_stack([mm.ravel(), nn.ravel()])
data = lg.lattice_orders(A, labels, wavelength0=wavelength0, incident=(.3,.4))

def mask(x, y):
    return ((abs((x/px+.5)%1-.5) < Dx/2) &
            (abs((y/py+.5)%1-.5) < Dy/2)).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for ax, count, title in zip(axes, [1, 3], ['One unit cell', 'Repeated pattern (drawing only)']):
    x = np.linspace(-count*px/2, count*px/2, 600, endpoint=False)
    y = np.linspace(-count*py/2, count*py/2, 600, endpoint=False)
    ax.imshow(mask(x[None,:], y[:,None]), origin='lower', cmap='gray', vmin=0, vmax=1,
              extent=[x[0], count*px/2, y[0], count*py/2], interpolation='nearest')
    ax.set(title=title, xlabel='x (um)', ylabel='y (um)', aspect='equal')
plt.show()


White is unit amplitude transmission; black is zero. The finite patch is an illustration of an **infinite periodic** mask. It adds no finite-aperture envelope or peak broadening to this calculation.

## 2. Complete each wavevector on the outgoing branch

In the outgoing medium, $\mathbf k\cdot\mathbf k=(n_{\rm out}k_0)^2$. Therefore
$$\boxed{k_{z,mp}=-\sqrt{(n_{\rm out}k_0)^2-|\mathbf k_{\parallel,mp}|^2}}.$$
For positive radicand the negative real root travels into $z<0$ under $e^{-i\omega t}$. Zero is **grazing**: no normal propagation. For negative radicand the principal square root followed by a minus sign gives $k_z=-i\kappa$, $\kappa>0$. At $z=-s$,
$$e^{ik_z z}=e^{-\kappa s},\qquad s\ge0.$$
Thus the field decays away from the grating in the transmitted region. When completing an evanescent wavevector, keep the algebraic squares in the dispersion equation; replacing them with a Hermitian norm would change the equation. An evanescent component has no real direction-cosine vector.

Refractive indices determine the incident translation, outgoing directions $\mathbf k/(n_{\rm out}k_0)$, and propagation disk radius $n_{\rm out}k_0$. Physical periods alone determine $\mathbf b_j$.


## 3. The same lattice on three coordinate scales

Use the outgoing medium for all three views. The transverse propagation boundaries have radii $n_{\rm out}k_0$ in wavevector coordinates, $n_{\rm out}/\lambda_0$ in cycles per micrometre, and 1 in direction cosines. Equality is grazing. The increments are $2\pi/p_x,2\pi/p_y$; $1/p_x,1/p_y$; and $\lambda_0/(n_{\rm out}p_x),\lambda_0/(n_{\rm out}p_y)$ respectively. None of these disks is an optical pupil.

**Predict:** with the stated oblique illumination, does $(1,1)$ propagate? Does testing each transverse component separately suffice?


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(13,4), constrained_layout=True)
for ax, scale in zip(axes, ['wavevector','frequency','direction']):
    coordinate_plot(ax, data, scale, annotate=(scale=='direction'))
axes[0].legend(loc='upper left', fontsize=8)
plt.show()
j = np.flatnonzero(np.all(labels==(1,1),axis=1))[0]
print('(1,1):', data['components'][j], data['status'][j])


:::{dropdown} Answer

$(1,1)$ has transverse normalized components $(0.8,0.8)$. Their squared sum is 1.28, so the order is evanescent even though each component is smaller than 1. Its longitudinal component is $-i\sqrt{0.28}$. A point outside the direction disk is a candidate label, not a real ray.

:::

## 4. Translate, rescale and rotate the lattice

Changing incidence translates every candidate by the same transverse vector. Changing $p_x$ changes only the corresponding reciprocal increment. Rotation changes the basis vectors and the associated physical directions; order labels remain integers in the rotating basis. Here all panels use outgoing direction-cosine coordinates with the same scale.


In [ ]:
angle = np.deg2rad(30)
R = np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]])
fig, axes = plt.subplots(1,4, figsize=(14,3.7), constrained_layout=True)
for ax, direct, inc, title in zip(axes,
        [A,np.diag([6.,py]),A,R@A],
        [(0.,0.),(0.,0.),(.3,.4),(0.,0.)],
        ['Normal incidence','Larger x pitch','Oblique incidence','Rotated 30 degrees']):
    d = lg.lattice_orders(direct, labels, wavelength0=wavelength0, incident=inc)
    coordinate_plot(ax,d,title=title)
plt.show()


## 5. Recover the one-dimensional symmetry

A grating continuously uniform along $y$ is unchanged by **every** $\Delta y$. Thus $e^{i(k_y-k_{y,\rm inc})\Delta y}=1$ for every displacement, forcing $k_y=k_{y,\rm inc}$. Only $m$ remains as an independent order label. This is the conically illuminated row in 5.1.

Increasing $p_y$ indefinitely instead makes reciprocal candidates densely spaced; it does not establish continuous uniformity or force their excitation to vanish. For a transmission independent of $y$, Fourier coefficients of every $p\ne0$ vanish. A symmetry-allowed location can likewise have zero amplitude in a genuinely two-dimensional structure. Details of rectangular and nonseparable cell coefficients are in [9.2: unit-cell coefficient derivations](9_2_Sampling_Complex_Fields_and_Their_Spectra.ipynb#cell-coefficients-9-2).

### Small calculation: follow an evanescent order into the output medium

For the stated grating and oblique illumination, order $(1,1)$ has normalized transverse components $(0.8,0.8)$. Since $0.8^2+0.8^2>1$, its longitudinal component is imaginary. At depth $s=-z$, its amplitude relative to the surface is $|e^{-ik_zs}|$.

**Predict, then run:** does the amplitude grow or decay as we move farther into $z<0$? Change the order to $(0,0)$ and compare. This calculation follows one prescribed component; it does not determine its surface excitation.


In [ ]:
order = (1, 1)
j = np.flatnonzero(np.all(data['labels'] == order, axis=1))[0]
kz = data['k'][j, 2]
depths = np.array([0., .5, 1.])  # micrometres into z < 0
relative_amplitude = np.abs(np.exp(-1j*kz*depths))
print(f'Order {order}: {data["status"][j]}, kz = {kz:.4f} rad/um')
for depth, amplitude in zip(depths, relative_amplitude):
    print(f'Depth {depth:.1f} um: amplitude / surface amplitude = {amplitude:.4f}')


## Final checkpoint

What selects the two integer labels? How does this differ from conical incidence? What changes when the outgoing index changes?

:::{dropdown} Answers

Two independent discrete translations permit two independent integer phase cycles. Continuous uniformity along grooves instead conserves that component, even at conical incidence. Changing the outgoing index changes the propagation boundary and directions, while reciprocal increments remain set by the structure.

:::

We now have order directions without introducing a collecting system. Which of these directions can an objective accept, and when do angular labels correspond to physical pupil positions? Continue to [5.4: From diffraction coordinates to pupils](5_4_From_Diffraction_Coordinates_to_Pupils.ipynb).

[Previous: 5.2](5_2_Direction_Cosines_and_Spatial_Frequency_Coordinates.ipynb) · [Chapter 5](5_0_Grating_Diffraction_in_Three_Dimensions.ipynb)

For the translation and outgoing-wave foundations, see [5.1's references](5_1_Three_Dimensional_Orders_and_Conical_Diffraction.ipynb#ref-5-1-maystre); for the coordinate convention, see [5.2](5_2_Direction_Cosines_and_Spatial_Frequency_Coordinates.ipynb). The two-period rule above follows from applying the same translation argument independently in two directions.


## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete [scripts download](Downloads.md) to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. Two translation symmetries and two integers

**Script:** [5_3_01_reciprocal_lattice_basis.py](Wave_solver_py_scripts/scripts/chapter_5/5_3_01_reciprocal_lattice_basis.py)

Construct a reciprocal basis for a nonorthogonal real-space lattice.

### 2. Two-dimensional diffraction orders

**Script:** [5_3_02_two_dimensional_order_lattice.py](Wave_solver_py_scripts/scripts/chapter_5/5_3_02_two_dimensional_order_lattice.py)

Classify a two-integer order lattice against the output propagation disk.

### 3. Translate rescale and rotate the lattice

**Script:** [5_3_03_rotate_lattice.py](Wave_solver_py_scripts/scripts/chapter_5/5_3_03_rotate_lattice.py)

Rotate direct and reciprocal lattices consistently.

### 4. Follow an evanescent order into the output medium

**Script:** [5_3_04_evanescent_lattice_order.py](Wave_solver_py_scripts/scripts/chapter_5/5_3_04_evanescent_lattice_order.py)

Propagate one evanescent two-dimensional order.
